## Setup

In [1]:
from pyspark.sql import Row, SparkSession

spark = (
    SparkSession.builder.appName("sample_spark_session")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/26 15:11:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
from pathlib import Path
from generate_data import run
from run_ddl import run_ddl

run(scaling_factor=0.01, format="csv", output_folder_path="./data") # data will be nGB on your disk\n",
run_ddl(data_path=Path("./data"), spark=spark, recreate=True)

2026-06-26 15:14:14,149 - generate_data - INFO - Starting TPC-H dataset generation pipeline


2026-06-26 15:14:14,150 - generate_data - INFO - Starting TPC-H table creation with scaling factor 0.01


2026-06-26 15:14:15,652 - generate_data - INFO - TPC-H tables created with scaling factor 0.01


2026-06-26 15:14:15,652 - generate_data - INFO - Starting export of TPC-H tables to csv format in ./data


2026-06-26 15:14:15,656 - generate_data - INFO - Found 8 tables to export


...

2026-06-26 15:14:25,646 - run_ddl - INFO - Reading part data from data/part.csv


2026-06-26 15:14:26,122 - run_ddl - INFO - Reading partsupp data from data/partsupp.csv


2026-06-26 15:14:26,677 - run_ddl - INFO - Reading region data from data/region.csv


2026-06-26 15:14:27,139 - run_ddl - INFO - Reading supplier data from data/supplier.csv


2026-06-26 15:14:27,594 - run_ddl - INFO - Reading dim_date data from dim_date.csv


In [5]:
spark.sql("use prod.db")
spark.sql("show tables").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  prod.db| partsupp|      false|
|  prod.db| supplier|      false|
|  prod.db|     part|      false|
|  prod.db|   orders|      false|
|  prod.db| dim_date|      false|
|  prod.db| customer|      false|
|  prod.db|   nation|      false|
|  prod.db| lineitem|      false|
|  prod.db|   region|      false|
+---------+---------+-----------+



#  Design Patterns: Well designed fact and dimension pipelines will take care of 90% of your job 

## Facts and dimensions are the most critical data products DE create [10 min]

[5 min]
- add: image 
- raw -> bronze: tools: fivetran, kafka -> S3 connector, CDC, dlt, etc
- mart/gold: semantic layer, metrics layer, views, JOINS + GROUP BYs
- some companies use a inmon layer pre-kimball; usually at companies where there are mutliple organizations and there is a need to combien them into a unified data model
- most companies follow the multi hop architecture, it can take slightly different forms + different naming (bronze/silver/gold, raw/base/stage/clean, marts, etc)
- essentially the idea is
    1. Bring upstream data into a cloud store 
    2. Create a data type and naming convention confirming dataset of the raw data called bronze (some companies combine 1 & 2, some create 2 as views on top of 1)
    3. [Optional] Create a 3nf to unify data from multiple systems. This is not very common, but prevelant at companies where data team is expected to unify data from multiple organizations (think orgs that were acquired over time)
    4. The Kimball layer; Data is modelled as facts, dims and bridge tables. Commonly referred to as star schema. Highly denormalized, ie keep number of tables low by combining multiple upstream tables to a few dimension tables.
    5. well designed Facts and dimensions are able to answer any question about a business, given that the data is collected. However this often involves joining them and groupin them, while calculating the right metrics. The problem is that the SQL knowledge and metric computation knowledge are hard for most non technical people. So DEs often create marts/summary/pre-aggregated tables, which are basically joined + grouped versions of the raw fact and dim tables. There is also metric layers (lookml, etc) where DEs define the dims and metrics as code and users can drag-and-drop them to create dashboards. Lately there is also the semantic layer which aims to enable AI agents to write the correct queries in response to user questions.
- The messy middle, which gives DE the leverage to asymmetrically impact outcomes is the kimball layer. 
- The ease and reliability of using data depends on the kimball layer. 
- In this workshop we will go over the patterns to design your fact and dim pipelines. 
- By the end of this post you will have a flowchart that you can use to quickly define you fact and dim tables for your use case.
- add: image showing dump -> bronze -> silver -> gold , with highlight around bronze layer and tools/teams under the bronze and gold layers

[5 min]
Q & A

**Additional Links**

1. Fivetran 
2. CDC with debezium 
3. OSI semantic layer

TL;DR [5 min]

add: image representing the fact/dim design patterns

## The Fact Pipeline Design: Read input incrementally, Left Join mapping data & create new rows, INSERT OVERWRITE BY PARTITION to output [25 m]

- add: image of extracting orders data based on time, enriching orders table with mapping data (dim_date), and loading with insert overwrite to output 
#### Exercise script to do this [5 min], use the table above
- Explain answers [5 min]


In [7]:
spark.sql("select min(o_orderdate), max(o_orderdate) from prod.db.orders").show()

+----------------+----------------+
|min(o_orderdate)|max(o_orderdate)|
+----------------+----------------+
|      1992-01-01|      1998-08-02|
+----------------+----------------+



In [14]:
spark.sql("select min(full_date), max(full_date) from prod.db.dim_date").show()

+--------------+--------------+
|min(full_date)|max(full_date)|
+--------------+--------------+
|    1990-01-01|    2001-12-31|
+--------------+--------------+



In [9]:
! uv run ./fct_orders.py --start-time 1995-01-01 --end-time 1996-01-01

2026-06-26 15:22:10,254 - py4j.clientserver - INFO - Closing down clientserver connection


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/26 15:22:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/06/26 15:22:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [13]:
spark.table("local.silver.fct_orders").limit(2).toPandas()

,o_orderkey,o_custkey,o_orderstatus,o_totalprice,o_orderdate,o_orderpriority,o_clerk,o_shippriority,o_comment,date_key,year,quarter,quarter_name,month,month_name,day_of_month,day_name,week_of_year,iso_week,is_weekend,is_holiday,holiday_name
0,3302,334,O,51365.63,1995-11-14,2-HIGH,Clerk#000000367,0,te closely between the blithely pending asymptotes? quickly re,19951114,1995,4,Q4,11,November,14,Tuesday,46,46,False,False,None
1,16260,781,O,50156.58,1995-11-14,2-HIGH,Clerk#000000002,0,quests integrate furiously fluffily regular pinto beans-- quickly pending th,19951114,1995,4,Q4,11,November,14,Tuesday,46,46,False,False,None



[5 min]
- Most fact table involves enrichment, ie. COALESCE + LEFT JOIN, creating new column,
- However if the columns created encoded business logic, they should be pushed to the application layer
- for example: a column computing cost after discount is valid, as its using existing logic 
- however if you find yourself changing the discount % based on a product id, because `we didn't do it in the backend` you have to push the application team to implement it 
- date range [str & end)
- alt created_at > max(created_at) from output

[10 min]
- add: image of transformation types
- in addition to standard enrichement style transformation, there are common patterns where you will need to do window function or self join or fact-fact join, let's look at some example
- Sessionization/attributions: idea is to tag related rows. Note that event order plays a key role here. I.e. you can't attribute a conversion to a click if you converstion row arrives and click hasnt (note: late arriving data), typicaly the reason people wait a few hours before processing data. This pattern typically involves window functions, however in complex cases the
- add: code example
- deduplication: either caused how data is dumped into the bronze layer or how your system works. For example if a user accidentally double clicks on a purchase button do you record it as 2 purchases or 1 purchase?
- add: code example
- UNION: Typical when you buy same type of data from multiple vendors you will need to combine them to have the same structure. For example you purchase behaviour data from Facebook and google, and have to combine it (naive example). These are flaky, especially with constantly changing input structures, and will often involve columns that are present in one but absent in another. when faced with such as situation, look for industry standards and match your data to it. For example, [OpenRTB](https://github.com/InteractiveAdvertisingBureau/openrtb2.x/blob/main/2.6.md#3233---object-refresh-) is a common standard used by most ads companies.
- DIFF: Common when ingesting data from vendors. Typically you identify the diff rows and process them. You may wonder why not just process the entire data, this may be due to various reasons, such as the vendor only sends data from past n days, but you need to keep track of history. The vendor data is not good quality so you need to check that the changes make sense. Your system can't handle the amount of data to process if you didnt diff it.
- add: links for further reading; adv sql, sql 25 tips, 

### Example 
copy over duplication problem from https://github.com/josephmachado/adv_data_transformation_in_sql/blob/main/concepts/query_templates/query_template.ipynb

ADD: feedback link and possible sign up for hands-on data engineering design pattern book (title: TBD)

## Dim pipeline design: select * read, inner join, overwrite table [10 m]

- most dim pipelines read full inputs as sources and inner join them to create denormalized tables

#### example [5 min]; why inner join v left join for completeness 

- the key things to be aware of here are the completeness of input data
- you can use a left join, however this will flow downstream and impact the fact tabes

In [24]:
spark.sql("select c_mktsegment, count(*) from customer group by 1").show()

+------------+--------+
|c_mktsegment|count(1)|
+------------+--------+
|   MACHINERY|     288|
|  AUTOMOBILE|     302|
|    BUILDING|     337|
|   HOUSEHOLD|     294|
|   FURNITURE|     279|
+------------+--------+



In [27]:
spark.sql("drop table if exists prod.db.dim_mktsegment");

spark.sql("""
    CREATE TABLE IF NOT EXISTS prod.db.dim_mktsegment (
      c_mktsegment        STRING,
      segment_description STRING,
      priority_tier       STRING
    ) USING iceberg
    TBLPROPERTIES (
      'format-version' = '2'
    )
    """);

spark.sql("""
    INSERT INTO prod.db.dim_mktsegment VALUES
      ('MACHINERY',  'Industrial machinery and equipment buyers', 'High'),
      ('AUTOMOBILE', 'Automotive and vehicle-related customers',   'High'),
      ('BUILDING',   'Construction and building materials sector', 'Medium'),
      ('HOUSEHOLD',  'Household goods and consumer products',      'Medium')
    """)

DataFrame[]

In [28]:
spark.sql("""
select c.*
, s.segment_description
, s.priority_tier
from prod.db.customer c
join prod.db.dim_mktsegment s
on c.c_mktsegment = s.c_mktsegment
where c.c_mktsegment = 'FURNITURE'
""").show(5)

+---------+------+---------+-----------+-------+---------+------------+---------+-------------------+-------------+
|c_custkey|c_name|c_address|c_nationkey|c_phone|c_acctbal|c_mktsegment|c_comment|segment_description|priority_tier|
+---------+------+---------+-----------+-------+---------+------------+---------+-------------------+-------------+
+---------+------+---------+-----------+-------+---------+------------+---------+-------------------+-------------+



In [32]:
spark.sql("""
with dim_customer as (select c.*
, s.segment_description
, s.priority_tier
from prod.db.customer c
join prod.db.dim_mktsegment s
on c.c_mktsegment = s.c_mktsegment
)
select c_mktsegment, segment_description, count(*) from dim_customer group by 1, 2
""").show(5)

+------------+--------------------+--------+
|c_mktsegment| segment_description|count(1)|
+------------+--------------------+--------+
|    BUILDING|Construction and ...|     337|
|   MACHINERY|Industrial machin...|     288|
|   HOUSEHOLD|Household goods a...|     294|
|  AUTOMOBILE|Automotive and ve...|     302|
+------------+--------------------+--------+



* inner join lost us the `FURNITURE`
* this may be ok, but be mindful of the choice you make
* you canalso use COALESE + LEFT JOIN as shown below or decide to wait or decide that the data will cathch up in the next run
* if you chose to use inner join, you will need to ensure that the downstream users know that the data may not be complete. 
* add image showing options as diverging paths, and their cons

In [33]:
spark.sql("""
with dim_customer as (select c.*
, COALESCE(s.segment_description, 'UNKNOWN') as segment_description
, COALESCE(s.priority_tier, 'UNKNOWN') as priority_tier
from prod.db.customer c
left join prod.db.dim_mktsegment s
on c.c_mktsegment = s.c_mktsegment
)
select c_mktsegment, segment_description, count(*) from dim_customer group by 1, 2
""").show(5)

+------------+--------------------+--------+
|c_mktsegment| segment_description|count(1)|
+------------+--------------------+--------+
|    BUILDING|Construction and ...|     337|
|   MACHINERY|Industrial machin...|     288|
|   FURNITURE|             UNKNOWN|     279|
|   HOUSEHOLD|Household goods a...|     294|
|  AUTOMOBILE|Automotive and ve...|     302|
+------------+--------------------+--------+




#### exercise [5 min] run this pipeline, chek the output, what is wrong?  -- combine with next
A: incomplete inner join
- explain: data completeness is an issue, you need to know when to run this pipeline (when inputs are complete) or wait for the next run to catch missing data. 


In [45]:
spark.sql("drop table if exists prod.db.brand")
spark.sql("""
    CREATE TABLE IF NOT EXISTS prod.db.brand (
      brand_id    BIGINT,
      brand_name  STRING,
      country     STRING,
      created_at  TIMESTAMP,
      updated_at  TIMESTAMP
    ) USING iceberg
    TBLPROPERTIES (
      'format-version' = '2'
    )
    """)

spark.sql("drop table if exists prod.db.product")
spark.sql("""
    CREATE TABLE IF NOT EXISTS prod.db.product (
      product_id    BIGINT,
      product_name  STRING,
      brand_id      BIGINT,   -- FK -> brand.brand_id (1:1)
      price         DECIMAL(15,2),
      created_at    TIMESTAMP,
      updated_at    TIMESTAMP
    ) USING iceberg
    TBLPROPERTIES (
      'format-version' = '2'
    )
    """)

spark.sql("""
    INSERT INTO prod.db.brand VALUES
      (1, 'Acme',     'USA',     TIMESTAMP '2024-01-10 09:00:00', TIMESTAMP '2024-01-10 09:00:00'),
      (2, 'Globex',   'Germany', TIMESTAMP '2024-01-12 10:30:00', TIMESTAMP '2024-02-01 14:15:00'),
      (3, 'Initech',  'Canada',  TIMESTAMP '2024-01-15 11:45:00', TIMESTAMP '2024-01-15 11:45:00'),
      (4, 'Umbrella', 'Japan',   TIMESTAMP '2024-01-20 08:20:00', TIMESTAMP '2024-03-05 16:00:00')
    """)

spark.sql("""
    INSERT INTO prod.db.product VALUES
      (101, 'Cordless Drill',     1, 79.99,  TIMESTAMP '2024-01-11 09:00:00', TIMESTAMP '2024-01-14 12:00:00'),
      (102, 'Circular Saw',       1, 129.50, TIMESTAMP '2024-01-11 14:30:00', TIMESTAMP '2024-01-11 14:30:00'),
      (103, 'Smart Thermostat',   2, 199.00, TIMESTAMP '2024-01-13 10:00:00', TIMESTAMP '2024-01-18 16:20:00'),
      (104, 'LED Bulb 4-Pack',    2, 24.99,  TIMESTAMP '2024-01-14 11:15:00', TIMESTAMP '2024-01-14 11:15:00'),
      (105, 'Office Chair',       3, 149.95, TIMESTAMP '2024-01-16 08:45:00', TIMESTAMP '2024-01-19 13:00:00'),
      (106, 'Standing Desk',      3, 389.00, TIMESTAMP '2024-01-16 15:30:00', TIMESTAMP '2024-01-16 15:30:00'),
      (107, 'Air Purifier',       4, 159.99, TIMESTAMP '2024-01-20 09:10:00', TIMESTAMP '2024-01-20 09:10:00'),
      (108, 'Humidifier',         4, 59.99,  TIMESTAMP '2024-01-20 10:40:00', TIMESTAMP '2024-01-20 17:00:00'),
      (109, 'Espresso Machine',   1, 249.00, TIMESTAMP '2024-01-12 13:00:00', TIMESTAMP '2024-01-15 09:30:00'),
      (110, 'Robot Vacuum',       2, 299.99, TIMESTAMP '2024-01-15 13:30:00', TIMESTAMP '2024-01-17 10:45:00')
    """)

DataFrame[]

In [46]:
spark.sql("""
    SELECT
      p.product_id,
      p.product_name,
      p.price,
      p.brand_id,
      b.brand_name,
      b.country,
      p.created_at  AS product_created_at,
      p.updated_at  AS product_updated_at,
      b.created_at  AS brand_created_at,
      b.updated_at  AS brand_updated_at
    FROM prod.db.product p
    JOIN prod.db.brand b
      ON p.brand_id = b.brand_id
    ORDER BY p.product_id
    """).show(2)

+----------+--------------+------+--------+----------+-------+-------------------+-------------------+-------------------+-------------------+
|product_id|  product_name| price|brand_id|brand_name|country| product_created_at| product_updated_at|   brand_created_at|   brand_updated_at|
+----------+--------------+------+--------+----------+-------+-------------------+-------------------+-------------------+-------------------+
|       101|Cordless Drill| 79.99|       1|      Acme|    USA|2024-01-11 09:00:00|2024-01-14 12:00:00|2024-01-10 09:00:00|2024-01-10 09:00:00|
|       102|  Circular Saw|129.50|       1|      Acme|    USA|2024-01-11 14:30:00|2024-01-11 14:30:00|2024-01-10 09:00:00|2024-01-10 09:00:00|
+----------+--------------+------+--------+----------+-------+-------------------+-------------------+-------------------+-------------------+
only showing top 2 rows


In [47]:
spark.sql("select * from product where date(updated_at) = '2024-01-17' ").show()

+----------+------------+--------+------+-------------------+-------------------+
|product_id|product_name|brand_id| price|         created_at|         updated_at|
+----------+------------+--------+------+-------------------+-------------------+
|       110|Robot Vacuum|       2|299.99|2024-01-15 13:30:00|2024-01-17 10:45:00|
+----------+------------+--------+------+-------------------+-------------------+



In [48]:
spark.sql("select * from brand where date(updated_at) = '2024-01-17' ").show()

+--------+----------+-------+----------+----------+
|brand_id|brand_name|country|created_at|updated_at|
+--------+----------+-------+----------+----------+
+--------+----------+-------+----------+----------+



In [50]:
spark.sql("""
    with product_incremental as (select * from prod.db.product where date(updated_at) = '2024-01-17'),
    brand_incremental as (select * from prod.db.brand where date(updated_at) = '2024-01-17')
    SELECT
      p.product_id,
      p.product_name,
      p.price,
      p.brand_id,
      b.brand_name,
      b.country,
      p.created_at  AS product_created_at,
      p.updated_at  AS product_updated_at,
      b.created_at  AS brand_created_at,
      b.updated_at  AS brand_updated_at
    FROM product_incremental p
    JOIN brand_incremental b
      ON p.brand_id = b.brand_id
    ORDER BY p.product_id
    """).show(2)

+----------+------------+-----+--------+----------+-------+------------------+------------------+----------------+----------------+
|product_id|product_name|price|brand_id|brand_name|country|product_created_at|product_updated_at|brand_created_at|brand_updated_at|
+----------+------------+-----+--------+----------+-------+------------------+------------------+----------------+----------------+
+----------+------------+-----+--------+----------+-------+------------------+------------------+----------------+----------------+




#### exercise [10m] read this code, what is wrong? Why not incremental dim? 
- A: this chunks data points to other inputs old chunks data => missed data
- dim tables are often small (a few million rows) and full select is often the simplest to maintain



Note:
- data asset driven pipeline to ensure data completeness add:link
- scd2: IFF users require history. Most users don't know how to use scd2 or care. Be mindful SCD2's are deceptively complext. add screenshot example

## Highlevel next steps

* failure analysis: missing data, bad schema, rerun on fail, defensive programming, input validation, scale (reduce data shuffle & pratition prune)
 

## Recap

* add: table image
* share link to article 